# Weather Data Exploration
This notebook analyzes the retrieved weather data and identifies potential data quality issues.

## 1. Setup
Prepare the environment by importing necessary libraries and setting up input/output directories.

In [ ]:
import os
import ast
import pandas as pd
import notebook_const

from src.const import WEATHER_RAW_DIR, WEATHER_PROCESSED_DIR

In [ ]:
os.makedirs(WEATHER_RAW_DIR, exist_ok=True)

## 2. Load Raw Weather Data
Load and combine all raw weather data files into a single DataFrame for analysis.

In [ ]:
# Load all raw weather data files
data_list = os.listdir(WEATHER_RAW_DIR)
weather = pd.DataFrame()
for file in data_list:
    if file.endswith('.parquet'):
        file_path = os.path.join(WEATHER_RAW_DIR, file)
        temp_df = pd.read_parquet(file_path)
        if not weather.empty:
            weather = pd.concat([weather, temp_df], ignore_index=True)
        else:
            weather = temp_df

print(f"Total records loaded: {len(weather)}")


In [ ]:
weather.head()

In [ ]:
weather.info()

## 3. Check Missing Values by Station
Analyze missing values for key features grouped by station. Identify stations with incomplete or insufficient data, such as missing rainfall measurements.

In [ ]:
cols_to_check = ['vis', 'rnfl_amt_pst1mt', 'rnfl_amt_pst1hr', 'air_temp', 'rel_hum', 'dwpt_temp', 'avg_wnd_spd_10m_pst10mts', 'avg_wnd_spd_10m_pst1hr']

# Calculate the number of null values for each column grouped by station name
result = weather.groupby('stn_nam-value')[cols_to_check].apply(lambda x: x.isnull().sum())
result['total_rows'] = weather.groupby('stn_nam-value').size()
result[['total_rows']+cols_to_check]

## 4. Identify Stations to Exclude
Based on the analysis, identify stations to exclude. Since rainfall data is critical for this project, stations with no rainfall measurements are carefully excluded.

In [ ]:
# Identify stations where rnfl_amt_pst1hr is all null
rain_null_check = weather.groupby('stn_nam-value')['rnfl_amt_pst1hr'].apply(lambda x: x.isnull().all())
exclude_stations = rain_null_check[rain_null_check].index.tolist()
exclude_stations = list(set(exclude_stations))

print(f"Stations to exclude (no rainfall data): {exclude_stations}")

In [ ]:
# Filter out excluded stations
weather_filtered = weather[~weather['stn_nam-value'].isin(exclude_stations)]
print(f"Records after filtering: {len(weather_filtered)}")

In [ ]:
# Calculate the number of null values for each column grouped by station name (after filtering)
result = weather_filtered.groupby('stn_nam-value')[cols_to_check].apply(lambda x: x.isnull().sum())
result['total_rows'] = weather_filtered.groupby('stn_nam-value').size()
result[['total_rows']+cols_to_check]

## 5. Time Bucket Analysis
Group the data into 10-minute intervals to ensure stable and consistent values for analysis.

In [ ]:
# Avoid SettingWithCopyWarning by using .loc
weather_filtered = weather_filtered.copy()

# Convert to datetime with UTC timezone
weather_filtered['date_tm-value'] = pd.to_datetime(weather_filtered['date_tm-value'], utc=True)

# Convert to local timezone
weather_filtered['local_time'] = weather_filtered['date_tm-value'].dt.tz_convert('America/Vancouver')
weather_filtered['time_bucket'] = weather_filtered['local_time'].dt.floor('10min')

weather_filtered[['date_tm-value', 'local_time', 'time_bucket']].head()

In [ ]:
# Check null values after aggregation by time bucket
weather_filtered.groupby(['stn_nam-value', 'time_bucket'])[cols_to_check].mean().isnull().sum()

## 6. Extract and Save Station Coordinates
Extract the longitude and latitude of each weather station to facilitate mapping and integration with other datasets.

In [ ]:
# Extract geometry info
def parse_coordinates(coord):
    """Parse coordinates from string, list, or numpy array."""
    if isinstance(coord, str):
        return ast.literal_eval(coord)
    elif hasattr(coord, 'tolist'):  # numpy array
        return coord.tolist()
    return coord  # already a list

weather_filtered['geometry.coordinates'] = weather_filtered['geometry.coordinates'].apply(parse_coordinates)
station_coords = weather_filtered.groupby('stn_nam-value')['geometry.coordinates'].first()
coords_df = pd.DataFrame(station_coords.tolist(), index=station_coords.index)
coords_df = coords_df.iloc[:, :2]
coords_df.columns = ['lon', 'lat']
coords_df.index.name = 'station_name'
coords_df = coords_df.reset_index()
coords_df

In [ ]:
import folium

# Create a map centered on Vancouver area
center_lat = coords_df['lat'].mean()
center_lon = coords_df['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

# Add markers for each station
for _, row in coords_df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=row['station_name'],
        tooltip=row['station_name'],
        icon=folium.Icon(color='blue', icon='cloud')
    ).add_to(m)

m

In [ ]:
coords_df.to_parquet(f'{WEATHER_PROCESSED_DIR}/station_coordinates_mst.parquet', index=False)
print(f"Station coordinates saved to {WEATHER_PROCESSED_DIR}/station_coordinates_mst.parquet")

## 7. Save Excluded Stations List
Save the list of excluded stations for reference and use in subsequent processing steps.

In [ ]:
# Save excluded stations list for use in processing notebook
exclude_df = pd.DataFrame({'station_name': exclude_stations})
exclude_df.to_parquet(f'{WEATHER_PROCESSED_DIR}/excluded_stations.parquet', index=False)
print(f"Excluded stations saved to {WEATHER_PROCESSED_DIR}/excluded_stations.parquet")